In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
injections = pd.read_csv('injections.dat', delimiter='\t')
injections.head()

In [ ]:
plt.figure(figsize=(16,18))
parameter_names = ['inclination', 'distance', 'mass1', 'mass2', 'spin1z', 'spin2z']

for i, param in enumerate(parameter_names):
    plt.subplot(3, 2, i+1)
    plt.hist(injections[param], bins=30, alpha=0.7, color='blue', edgecolor='black')
    mean = injections[param].mean()
    std = injections[param].std()
    plt.axvline(mean, color='red', linestyle='dashed', linewidth=1)
    # plt.axvline(mean + std, color='green', linestyle='dashed', linewidth=1)
    # plt.axvline(mean - std, color='green', linestyle='dashed', linewidth=1)
    plt.title(f'Distribution of {param}')
    plt.xlabel(param)
    plt.ylabel('Count')
    plt.grid()
plt.show()

In [ ]:
import healpy as hp
theta = injections['latitude'].values + np.pi/2
phi = injections['longitude'].values

nside = 32
npix = hp.nside2npix(nside)
sky_map = np.zeros(npix)
pixels = hp.ang2pix(nside, theta, phi)
for pix in pixels:
    sky_map[pix] += 1
hp.mollview(sky_map, title='Sky Distribution of Injections', unit='Number of Injections')
hp.graticule()

In [ ]:
from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import z_at_value
import numpy as np

import astropy.units as u

def z_to_comoving_volume(z, cosmology=cosmo):
    """
    将红移 z（标量或数组）转换为共动体积（返回 astropy Quantity，单位 Mpc^3）。
    """
    return cosmology.comoving_volume(z)

def comoving_volume_to_z(V, cosmology=cosmo, z_max=20.0, ngrid=20000):
    """
    将共动体积 V 转回红移。
    - V 可以是带单位的 astropy Quantity（例如 1e9 * u.Mpc**3）或裸浮点（视为 Mpc^3）。
    - 对标量使用 astropy.cosmology.z_at_value 精确反演；对数组使用在 [0, z_max] 上的插值近似反演。
    - 返回标量或 numpy 数组（与输入形状一致）。
    """
    # 规范为 Quantity（单位 Mpc^3）
    if not hasattr(V, 'unit'):
        V = V * u.Mpc**3
    else:
        V = V.to(u.Mpc**3)

    # 标量情况：用 z_at_value
    if np.ndim(V.value) == 0:
        return z_at_value(cosmology.comoving_volume, V)

    # 数组情况：在 z 网格上计算共动体积并插值
    z_grid = np.linspace(0.0, z_max, ngrid)
    V_grid = cosmology.comoving_volume(z_grid).to(u.Mpc**3).value
    V_vals = np.asarray(V.value)
    z_vals = np.interp(V_vals, V_grid, z_grid, left=np.nan, right=np.nan)
    return z_vals

# 示例
z_example = np.array([0.10, 0.5, 1.0, 2.0])
V_example = z_to_comoving_volume(z_example)
print("z -> V (Mpc^3):", V_example)
print("V -> z (invert):", comoving_volume_to_z(V_example))

In [ ]:
allsky = pd.read_csv('allsky.dat',delimiter='\t')
allsky.sort_values('coinc_event_id', inplace=True)
allsky.head()

In [ ]:
coincs = pd.read_csv('coincs.dat', delimiter='\t')
coincs.head()

绘制定位天区以及距离在injection以及skymap结果中的差别：部分低信噪比事件会导致距离定位偏差极大

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(allsky['snr'].values, range=[8,20], bins=30, alpha=0.7, color='b')
plt.title('SNR Distribution of Coincident Events')
plt.xlabel('SNR')
plt.ylabel('Count')
plt.grid()
plt.show()

In [ ]:
inj_dist = injections['distance'].values
allsky_dist = allsky['distmean'].values
allsky_diststd = allsky['diststd'].values
deviation = (allsky_dist - inj_dist) / allsky_diststd
plt.figure(figsize=(10,6))
plt.hist(deviation, bins=30, alpha=0.7, color='green', edgecolor='black')
plt.title('Deviation of Estimated Distance from True Distance')
plt.xlabel('Deviation(skymap-injection) (sigma units)')
plt.ylabel('Count')
plt.grid()

In [ ]:
snr = allsky['snr'].values
plt.figure(figsize=(10,6))
plt.scatter(deviation, snr, alpha=0.7, color='purple', edgecolor='black')
plt.title('Deviation vs SNR')
plt.xlabel('Deviation (sigma units)')
plt.ylabel('SNR')
plt.ylim((8,20))
plt.grid()

In [ ]:
# plot area(90) vs snr
area90 = allsky['area(90)'].values
snr = allsky['snr'].values
plt.figure(figsize=(10,6))
plt.scatter(area90, snr, alpha=0.7, color='g', edgecolor='black')
plt.title('Area(90) vs SNR')
plt.xlabel('Area(90) (square degree')
plt.ylabel('SNR')
plt.ylim((8,20))
plt.grid()

injection中距离包含过多极远的事件，进行裁剪，保留小于1000Mpc的injection

In [ ]:
index = np.where(injections['distance'].values < 1000)[0]
# index = np.where(np.abs(deviation)<=3)[0]
print(f"Number of injections before cut: {len(injections)}")
print(f"Number of injections after cut: {len(index)}")
injections1 = injections.iloc[index]
coincs1 = coincs.iloc[index]
allsky1 = allsky.iloc[index]

In [ ]:
injections1.reset_index(drop=True, inplace=True)
coincs1.reset_index(drop=True, inplace=True)
allsky1.reset_index(drop=True, inplace=True)
injections1.to_csv('injections_cleaned.csv', sep=',', index=False)
coincs1.to_csv('coincs_cleaned.csv', sep=',', index=False)  
allsky1.to_csv('allsky_cleaned.csv', sep=',', index=False)

In [ ]:
plt.figure(figsize=(16,18))
parameter_names = ['inclination', 'distance', 'mass1', 'mass2', 'spin1z', 'spin2z']

for i, param in enumerate(parameter_names):
    plt.subplot(3, 2, i+1)
    plt.hist(injections1[param], bins=30, alpha=0.7, color='blue', edgecolor='black')
    mean = injections1[param].mean()
    std = injections1[param].std()
    plt.axvline(mean, color='red', linestyle='dashed', linewidth=1)
    # plt.axvline(mean + std, color='green', linestyle='dashed', linewidth=1)
    # plt.axvline(mean - std, color='green', linestyle='dashed', linewidth=1)
    plt.title(f'Distribution of {param}')
    plt.xlabel(param)
    plt.ylabel('Count')
    plt.grid()
plt.show()

In [ ]:
# plot area(90) vs snr
area901 = allsky1['area(90)'].values
snr1 = allsky1['snr'].values
plt.figure(figsize=(10,6))
plt.scatter(area901, snr1, alpha=0.7, color='g', edgecolor='black')
plt.title('Area(90) vs SNR')
plt.xlabel('Area(90) (square degree')
plt.ylabel('SNR')
plt.ylim((8,20))
plt.grid()

In [ ]:
inj1_dist = injections1['distance'].values
allsky1_dist = allsky1['distmean'].values
allsky1_diststd = allsky1['diststd'].values
deviation1 = (allsky1_dist - inj1_dist) / allsky1_diststd
plt.figure(figsize=(10,6))
plt.hist(deviation1, bins=30, alpha=0.7, color='green', edgecolor='black')
plt.title('Deviation of Estimated Distance from True Distance(After cut)')
plt.xlabel('Deviation (sigma units)')
plt.ylabel('Count')
plt.grid()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(deviation1, snr1, alpha=0.7, color='purple', edgecolor='black')
plt.title('Deviation vs SNR')
plt.xlabel('Deviation (sigma units)')
plt.ylabel('SNR')
# plt.ylim((8,20))
plt.grid()